# 1. Setup
**Approach 3: CLIP vision-language model.** Every pair, seen or unseen, becomes a text prompt such as
"a photo of a tomato leaf with early blight". The image is matched to the closest prompt.
Unseen pairs need no training images: their crop and disease words are enough (zero-shot side information).
We fine-tune only the last image-encoder blocks on seen pairs, so CLIP keeps its broad, robust knowledge.
Backbone is CLIP ViT-B/16 (not resnet50) because the method needs a matching text encoder.

In [ ]:
import copy
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # lets the notebook import common/

import open_clip
import torch
import torch.nn as nn
import torch.nn.functional as F

from common.config import CHECKPOINT_DIR, DEVICE, NUM_SEEN_PAIRS, set_seed
from common.dataset import get_crop_and_disease_names, get_loaders, get_pairs
from common.evaluate import evaluate, val_accuracy

MODEL_NAME = "approach3_clip"
DEBUG = True  # True: small data, 1 epoch. Set False for the full run.
HP = {"backbone": "ViT-B-16", "pretrained": "openai", "lr": 1e-5, "weight_decay": 0.05,
      "epochs": 10, "batch_size": 64, "optimizer": "AdamW", "trainable_blocks": 4,
      "prompts": ["a photo of a {crop} leaf with {disease}.",
                  "a close-up photo of a {crop} plant showing {disease}.",
                  "{disease} on a {crop} leaf."]}
EPOCHS = 1 if DEBUG else HP["epochs"]
set_seed(42)

# 2. Data

In [ ]:
train_loader, val_loader, seen_loader, unseen_loader = get_loaders(
    HP["batch_size"], norm="clip", debug=DEBUG)  # CLIP was trained with its own mean and std
pairs = get_pairs()
crop_names, disease_names = get_crop_and_disease_names()

# 3. Model

In [ ]:
clip_model = open_clip.create_model(HP["backbone"], pretrained=HP["pretrained"]).to(DEVICE)
tokenizer = open_clip.get_tokenizer(HP["backbone"])


@torch.no_grad()
def encode_pair_prompts():
    """One text feature per pair: the mean of all prompt templates (prompt ensembling)."""
    pair_features = []
    for crop_id, disease_id in pairs.tolist():
        crop, disease = crop_names[crop_id].replace("_", " "), disease_names[disease_id].replace("_", " ")
        prompts = [template.format(crop=crop, disease=disease) for template in HP["prompts"]]
        features = F.normalize(clip_model.encode_text(tokenizer(prompts).to(DEVICE)).float(), dim=1)
        pair_features.append(F.normalize(features.mean(dim=0), dim=0))
    return torch.stack(pair_features)  # [333, embed_dim]


text_features = encode_pair_prompts()  # the text encoder is frozen, so encode once

# Freeze everything, then unfreeze only the last few image blocks and the output projection
for parameter in clip_model.parameters():
    parameter.requires_grad = False
visual = clip_model.visual
for module in [*visual.transformer.resblocks[-HP["trainable_blocks"]:], visual.ln_post]:
    for parameter in module.parameters():
        parameter.requires_grad = True
visual.proj.requires_grad = True
logit_scale = clip_model.logit_scale.exp().item()


def pair_logits(images, pair_text):
    image_features = F.normalize(clip_model.encode_image(images).float(), dim=1)
    return logit_scale * image_features @ pair_text.T


@torch.no_grad()
def predict(images):
    clip_model.eval()
    return pair_logits(images.to(DEVICE), text_features)


print("val top-1 before fine-tuning (pure zero-shot):", val_accuracy(predict, val_loader))

# 4. Training

In [ ]:
trainable = [parameter for parameter in clip_model.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=HP["lr"], weight_decay=HP["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()
best_val, best_state = -1.0, None

for epoch in range(EPOCHS):
    clip_model.train()
    for images, _, _, pair_labels in train_loader:
        images, pair_labels = images.to(DEVICE), pair_labels.to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
            # Softmax over seen prompts only, so unseen prompts are never pushed away during training
            loss = criterion(pair_logits(images, text_features[:NUM_SEEN_PAIRS]), pair_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    scheduler.step()
    val_top1 = val_accuracy(predict, val_loader)
    print(f"epoch {epoch + 1}: loss {loss.item():.3f}, val top-1 {val_top1:.2f}%")
    if val_top1 > best_val:  # keep the best epoch by val, never by test
        best_val, best_state = val_top1, copy.deepcopy(clip_model.state_dict())

clip_model.load_state_dict(best_state)

# 5. Evaluation

In [ ]:
results = evaluate(predict, seen_loader, unseen_loader, MODEL_NAME, HP, robust=True)

# 6. Save

In [ ]:
CHECKPOINT_DIR.mkdir(exist_ok=True)
torch.save({"model_name": MODEL_NAME, "state_dict": clip_model.state_dict(), "hp": HP,
            "text_features": text_features.cpu()}, CHECKPOINT_DIR / f"{MODEL_NAME}.pt")